# 08 — Diversity & distribution metrics (the collapse axis)

Model collapse shows up as **shrinking diversity** and **drift from the real distribution**.
These metrics quantify it — comparing a corpus to itself (reference-free) or a candidate
corpus against a reference (reference-based).

**Library focus:** `sentence-transformers` (embeddings), `vendi-score`, `prdc`, `mauve-text`,
`sacrebleu`.

In [ ]:
# Two toy corpora: a diverse "real" set vs. a collapsed (repetitive) "synthetic" set.
real = [
    "The river wound slowly through the green valley.",
    "Quantum computers exploit superposition and entanglement.",
    "She planted tomatoes and basil in the spring garden.",
    "The orchestra tuned their instruments before the concert.",
    "Glaciers carve deep valleys over thousands of years.",
    "He debugged the parser late into the night.",
]
synth = [
    "The cat sat on the mat.",
    "The cat sat on the mat today.",
    "The cat sat on the warm mat.",
    "A cat sat on the mat.",
    "The cat sat on the mat again.",
    "The cat sat on the soft mat.",
]

## Reference-free: distinct-n and Self-BLEU

Distinct-2 = unique bigrams / total bigrams (higher = more diverse). Self-BLEU scores each
text against the others (higher = more self-similar = *less* diverse) — note it's high for
the collapsed corpus.

In [ ]:
def distinct_n(texts, n=2):
    total, uniq = 0, set()
    for t in texts:
        toks = t.split()
        grams = list(zip(*[toks[i:] for i in range(n)]))
        total += len(grams)
        uniq.update(grams)
    return len(uniq) / total if total else 0.0


print(f"distinct-2  real={distinct_n(real):.3f}  synth={distinct_n(synth):.3f}")

In [ ]:
import numpy as np
from sacrebleu import sentence_bleu


def self_bleu(texts):
    scores = [sentence_bleu(t, [o for o in texts if o != t]).score for t in texts]
    return float(np.mean(scores)) / 100.0  # sacrebleu is 0-100


print(
    f"self-BLEU   real={self_bleu(real):.3f}  synth={self_bleu(synth):.3f}  (lower=diverse)"
)

## Embeddings → Vendi score

Embed with sentence-transformers (unit-normalized), then the **Vendi score** gives the
*effective number of distinct samples* — it drops as a corpus collapses.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
real_emb = embedder.encode(real, normalize_embeddings=True, convert_to_numpy=True)
synth_emb = embedder.encode(synth, normalize_embeddings=True, convert_to_numpy=True)
print("embedding shape:", real_emb.shape)

In [ ]:
from vendi_score import vendi

print(
    f"Vendi  real={vendi.score_dual(real_emb.astype(np.float64)):.3f}  "
    f"synth={vendi.score_dual(synth_emb.astype(np.float64)):.3f}  (higher=diverse)"
)

## Reference-based: prdc and MAUVE

**prdc** (precision/recall/density/coverage): recall & coverage fall as synthetic stops
covering the real distribution. **MAUVE** summarizes distributional similarity in one number
(higher = closer). Both compare synthetic *against* real.

In [ ]:
from prdc import compute_prdc

# nearest_k must be < number of samples per set
print(compute_prdc(real_emb, synth_emb, nearest_k=3))

In [ ]:
import mauve

out = mauve.compute_mauve(
    p_text=real, q_text=synth, device_id=0, max_text_length=256, verbose=False
)
print(f"MAUVE = {out.mauve:.3f}  (1.0 = identical distributions)")

> **Project pointer:** `llm_core.metrics.diversity` implements all of these plus unigram-KL,
> Fréchet distance, RBF-MMD, vocabulary size and tail mass; `compute_panel` runs the whole
> suite and `scripts/diversity.py` is the CLI (synthetic-vs-Gen-0 across recursive
> generations). `llm_replay.generation.validation.validate` adds per-sample quality gates +
> perplexity on top.